# Mineração - regras de associação (Apriori)
Rede de academias - Banco de Dados II

Objetivo: descobrir quais planos, serviços e produtos são comprados juntos na mesma matrícula.

Técnica: regras de associação com Apriori (biblioteca mlxtend).

Base: base_transacoes.csv, exportada do DW pelo script sql/olap/03_exportar_base_mineracao.sql. Cada linha é um item vendido em uma matrícula, e cada matrícula é tratada como uma cesta.

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 70)

base = pd.read_csv('base_transacoes.csv')
print(base.shape)
base.head(10)

(5571, 4)


,id_matricula,item,categoria,departamento
0,1,Armário Mensal,Comodidades,Serviços
1,1,Musculação Mensal,Musculação,Planos
2,2,Natação Trimestral,Natação,Planos
3,3,Camiseta Dry GG,Vestuário,Produtos
4,3,Creatina 150g,Suplementos,Produtos
5,3,Musculação Trimestral,Musculação,Planos
6,4,"Água Mineral 1,5L",Bebidas,Produtos
7,4,Dança Mensal,Dança,Planos
8,5,Avaliação Física,Avaliação,Serviços
9,5,Barra Proteica Cookies,Suplementos,Produtos


## 1. Preparação dos dados
Os planos aparecem em 4 durações (mensal, trimestral, semestral e anual) e alguns produtos em vários sabores ou tamanhos. Separados, cada um fica com suporte muito baixo, então agrupamos: os planos viram só a modalidade (ex.: Musculação Trimestral vira Plano Musculação), os pacotes de personal viram Personal Trainer e whey, creatina e luvas ficam sem sabor/tamanho.

Depois agrupamos por id_matricula (cada matrícula vira uma lista de itens) e usamos o TransactionEncoder para gerar a matriz booleana matrícula x item, que é a entrada do Apriori.

In [2]:
def generalizar(linha):
    item, categoria, departamento = linha['item'], linha['categoria'], linha['departamento']
    if departamento == 'Planos':
        return categoria if categoria.startswith('Plano') else 'Plano ' + categoria
    for prefixo, rotulo in [('Personal', 'Personal Trainer'), ('Whey', 'Whey Protein'),
                            ('Creatina', 'Creatina'), ('Luva de Treino', 'Luva de Treino'),
                            ('Luva de Boxe', 'Luva de Boxe')]:
        if item.startswith(prefixo):
            return rotulo
    return item

base['item_analise'] = base.apply(generalizar, axis=1)
cestas = base.groupby('id_matricula')['item_analise'].apply(lambda s: sorted(set(s))).tolist()

te = TransactionEncoder()
matriz = pd.DataFrame(te.fit(cestas).transform(cestas), columns=te.columns_)
print(f'{len(cestas)} matrículas (cestas) x {matriz.shape[1]} itens distintos')
print(f'Itens por matrícula: média {matriz.sum(axis=1).mean():.2f}, máximo {matriz.sum(axis=1).max()}')

1920 matrículas (cestas) x 66 itens distintos
Itens por matrícula: média 2.90, máximo 8


## 2. Parâmetros
- suporte mínimo = 0,02 (o conjunto precisa aparecer em pelo menos 2% das matrículas, umas 38 vendas)
- confiança mínima = 0,40
- lift mínimo = 1,2 (só associações acima do que seria esperado ao acaso)

In [3]:
SUPORTE_MIN, CONFIANCA_MIN, LIFT_MIN = 0.02, 0.40, 1.2

frequentes = apriori(matriz, min_support=SUPORTE_MIN, use_colnames=True)
frequentes['tamanho'] = frequentes['itemsets'].apply(len)
print(f'{len(frequentes)} conjuntos frequentes')
print(frequentes.groupby('tamanho').size().rename('quantidade').to_string())
frequentes.sort_values('support', ascending=False).head(10)

91 conjuntos frequentes
tamanho
1    34
2    38
3    17
4     2


,support,itemsets,tamanho
22,0.415625,frozenset({Plano Musculação}),1
3,0.281250,frozenset({Avaliação Física}),1
41,0.204167,"frozenset({Avaliação Física, Plano Musculação})",2
17,0.146354,frozenset({Personal Trainer}),1
24,0.113542,frozenset({Plano Pilates}),1
20,0.113021,frozenset({Plano Funcional}),1
30,0.109896,frozenset({Whey Protein}),1
64,0.109896,"frozenset({Whey Protein, Plano Musculação})",2
23,0.106771,frozenset({Plano Natação}),1
52,0.103646,"frozenset({Creatina, Plano Musculação})",2


In [4]:
regras = association_rules(frequentes, metric='confidence', min_threshold=CONFIANCA_MIN)
regras = regras[regras['lift'] >= LIFT_MIN].copy()
texto = lambda s: ', '.join(sorted(s))
regras['se_comprar'] = regras['antecedents'].apply(texto)
regras['entao_compra'] = regras['consequents'].apply(texto)
regras = regras[['se_comprar', 'entao_compra', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)
regras.round(4).to_csv('regras_associacao.csv', index=False)
print(f'{len(regras)} regras encontradas (salvas em regras_associacao.csv)')
regras.head(20).round(3)

110 regras encontradas (salvas em regras_associacao.csv)


,se_comprar,entao_compra,support,confidence,lift
24,Luva de Boxe,Protetor Bucal,0.027,0.689,23.215
25,Protetor Bucal,Luva de Boxe,0.027,0.895,23.215
80,Protetor Bucal,"Luva de Boxe, Plano Lutas",0.027,0.895,23.215
81,Luva de Boxe,"Plano Lutas, Protetor Bucal",0.027,0.689,23.215
77,"Plano Lutas, Protetor Bucal",Luva de Boxe,0.027,0.895,23.215
78,"Luva de Boxe, Plano Lutas",Protetor Bucal,0.027,0.689,23.215
97,Óculos de Natação,"Plano Natação, Touca de Natação",0.047,0.843,12.444
96,Touca de Natação,"Plano Natação, Óculos de Natação",0.047,0.700,12.444
92,"Plano Natação, Touca de Natação",Óculos de Natação,0.047,0.700,12.444
93,"Plano Natação, Óculos de Natação",Touca de Natação,0.047,0.843,12.444


## 3. Regras a partir do plano escolhido
Filtramos as regras em que o lado esquerdo é um plano, o personal ou a avaliação física, porque são as que a recepção consegue usar na hora da matrícula.

In [5]:
planos = set(base.loc[base['departamento'] == 'Planos', 'item_analise'])
acao_balcao = regras[regras['se_comprar'].isin(planos | {'Personal Trainer', 'Avaliação Física'})]
acao_balcao.round(3)

,se_comprar,entao_compra,support,confidence,lift
12,Plano Lutas,Bandagem para Lutas,0.057,0.616,10.847
22,Plano Lutas,Luva de Boxe,0.039,0.418,10.847
33,Plano Natação,Touca de Natação,0.068,0.634,9.366
95,Plano Natação,"Touca de Natação, Óculos de Natação",0.047,0.444,9.366
35,Plano Natação,Óculos de Natação,0.056,0.527,9.366
9,Plano Pilates,Avaliação Postural,0.050,0.440,8.807
5,Plano Musculação,Avaliação Física,0.204,0.491,1.747
4,Avaliação Física,Plano Musculação,0.204,0.726,1.747
3,Plano Funcional,Avaliação Física,0.051,0.447,1.589
6,Plano Total,Avaliação Física,0.027,0.421,1.499


In [6]:
A, C = 'Whey Protein', 'Creatina'
n = len(matriz)
n_A, n_C, n_AC = matriz[A].sum(), matriz[C].sum(), (matriz[A] & matriz[C]).sum()
suporte = n_AC / n
confianca = n_AC / n_A
lift = confianca / (n_C / n)
print(f'Matrículas: {n} | com {A}: {n_A} | com {C}: {n_C} | com ambos: {n_AC}')
print(f'suporte({A} -> {C})   = {n_AC}/{n} = {suporte:.4f}')
print(f'confiança({A} -> {C}) = {n_AC}/{n_A} = {confianca:.4f}')
print(f'lift({A} -> {C})      = {confianca:.4f} / ({n_C}/{n}) = {lift:.2f}')

Matrículas: 1920 | com Whey Protein: 211 | com Creatina: 199 | com ambos: 124
suporte(Whey Protein -> Creatina)   = 124/1920 = 0.0646
confiança(Whey Protein -> Creatina) = 124/211 = 0.5877
lift(Whey Protein -> Creatina)      = 0.5877 / (199/1920) = 5.67


## 4. Interpretação de negócio
1. Kit natação na matrícula. Entre as matrículas de natação, 63% levam touca e 53% levam óculos (lift 9,4). Quem leva óculos também leva touca em 84% dos casos (lift 12,4). Faz sentido oferecer um kit touca + óculos já no ato da matrícula, com preço fechado.
2. Kit lutador. Nas lutas, 62% das matrículas incluem bandagem (lift 10,8). Luva de boxe e protetor bucal saem juntos em 90% das vezes que o protetor é comprado (lift 23,2, a regra mais forte da base). Os acessórios de luta só são vendidos junto com o plano (confiança 100%), então a loja depende do momento da matrícula. Vale fazer campanha para alunos que já estão matriculados.
3. Suplementos andam em par. Quem compra whey compra creatina em 59% das matrículas (lift 5,7), e todo suplemento é vendido para aluno de musculação (confiança 100%). Um combo whey + creatina para o público de musculação aproveita um comportamento que já existe.
4. Avaliação física ainda tem espaço. A regra Musculação -> Avaliação Física tem o maior suporte da base (20% de todas as matrículas), mas confiança de 49%: metade dos alunos de musculação começa sem avaliação. Incluir a avaliação no plano, ou oferecê-la sempre na recepção, pode aumentar a receita de serviços.
5. Nutrição acompanha o personal. Todo plano alimentar personalizado foi vendido para quem contratou personal trainer (confiança 100%, lift 6,8). Um pacote personal + nutricionista formaliza essa venda casada.